# OpenKhmer v7 — **All recognition + math records, TRUE dual-T4 DDP**

This is the **full-corpus** alternative to the 120k-sample v6 fast-launch notebook. It does **not** apply a 25k-per-source / 30k-total / 120k-total training cap.

- **Target:** all **10,585,709** OCR-compatible training records from the completed OpenKhmer migration manifest (10,468,194 text recognition + 117,515 math recognition). Source-specific eligibility is still mandatory.
- **Two actual GPU workers:** Kaggle `GPU T4 x2`, `torchrun --nproc_per_node=2`, each rank handles **different TAR archives**; LoRA updates are synchronized using **DistributedDataParallel**, not `DataParallel` or mere GPU visibility.
- **No 100GB download/extraction:** stream one uncompressed image archive at a time from private Hugging Face (`dvdphobia/OpenKhmer`). Each rank has bounded background CPU image decoding; an **optional local TAR cache** or an already-existing `/tmp` mirror is supported.
- **Fast defaults:** 4-bit LoRA, FP16 on T4, image long-edge cap of 768px (may reduce tiny-text recognition accuracy), LoRA vision encoder frozen initially, optimizer with paged 8-bit states, gradient accumulation with only **one DDP sync per optimizer step**, periodic atomic checkpoints.
- **Resume:** each rank resumes from its own position using an immutable Hugging Face SHA and a fingerprinted partition plan; rank padding (at most one local-rank imbalance) is reported. No re-cleaning or re-exporting the datasets.
- **First run:** `SESSION_STEPS=100` is a deliberate **100-step benchmark**, *not* the full training cap. Set `SESSION_STEPS=0` after checking the speed to keep training from the last checkpoint; sessions will stop on a configurable wall-clock budget.

**Not part of this text-recognition objective:** 35,488 detection training images with polygon/box labels, plus held-out validation/test, `evaluation_only/` and text-only rows. Detection needs a dedicated localization/multitask recipe; do not mislabel boxes as OCR text. The source-rights/duplicate-leakage review from the earlier export is still outstanding, so this notebook **requires explicit source-rights acknowledgement** and permits an allowlist.

**GPU compatibility disclosure:** The TAR scheduler and full-corpus resume logic are unit-tested offline; **PaddleOCR-VL + Unsloth 4-bit + DDP on Kaggle's exact dual-T4 runtime has not been validated yet**. The first 100 steps are the required compatibility and throughput check. Do not mistake the notebook for a completed full-epoch training result.


## 1 — Enable GPU T4 ×2 and install compatible dependencies

In Kaggle Notebook settings choose **GPU T4 ×2**, turn **Internet ON**, and add Kaggle Secret `HF_TOKEN` with read permission to your private `dvdphobia/OpenKhmer` dataset. Optionally add a separate `HF_WRITE_TOKEN` for private checkpoint backups. Don't upgrade `huggingface-hub` to 2.x in a Kaggle environment whose `datasets` or `diffusers` need <2.0.

**If pip asks for a kernel restart, restart before running the remaining cells.** Never restart while a training job is actively running.


In [ ]:
%pip install -q --no-cache-dir "huggingface-hub>=1.32,<2" "datasets>=3,<5" "unsloth" "trl" "peft" "accelerate" "bitsandbytes" "pillow" "filelock" "fsspec"
import huggingface_hub
print('huggingface-hub:',huggingface_hub.__version__)
assert int(huggingface_hub.__version__.split('.')[0]) < 2, 'Restart the kernel after correcting huggingface-hub'


## 2 — Configure **all-corpus** training + persistence

The **full dataset** lives in Hugging Face. We recommend remote TAR streaming so Kaggle won't run out of `/tmp`. To reuse an already finished v5 full mirror in `/tmp`, set `USE_EXISTING_FULL_LOCAL_MIRROR=True` (the downloaded revision must match). `CACHE_ACTIVE_TAR_TO_TMP=True` is an optional middle ground that stages and deletes one TAR per GPU.

The first pass through 10.59 million images will not finish within 26 GPU-hours. Save checkpoints externally before the quota/session ends. The notebook's private Hugging Face backup is opt-in.

**License gate:** Your old export marked training/admission rights as not verified. Confirm permissions for each exact source, and use `APPROVED_SOURCES` to select only authorized sources. Don't treat file availability as a commercial-use license.


In [ ]:
from pathlib import Path
import os, json, shutil, subprocess, sys, time, hashlib

os.environ.setdefault('HF_XET_CHUNK_CACHE_SIZE_BYTES','0')
os.environ.setdefault('HF_XET_CACHE','/tmp/openkhmer_xet_v7')

HF_DATASET_REPO = 'dvdphobia/OpenKhmer'
HF_DATASET_REVISION = '10e03165be9dd1d10b4bc3b9688bfa1f3d075b1f'  # On a NEW Kaggle session paste the SHA printed in Step 5.
REQUIRE_COMPLETE_UPLOAD = True  # Needs all 630 final TAR archives for true full-corpus training.
EXPECTED_EXPORT_TARS = 630
INCLUDE_MATH_RECOGNITION = True

# None means all OCR-compatible sources, after you verify rights. Or list exact
# folder paths like {'recognition/SoyVitou_KhmerSynthetic1M-680e080d2f'}.
APPROVED_SOURCES = None
ACKNOWLEDGE_SOURCE_RIGHTS = True  # Set True ONLY after checking exact dataset permissions.

# Actual two-worker data parallelism. Default refuses 1 GPU rather than claiming x2.
REQUIRE_DUAL_T4 = True
MODEL_ID = 'unsloth/PaddleOCR-VL'
PER_GPU_BATCH = 2             # T4 16GB: if OOM, set 1 and GRAD_ACCUM_STEPS=4.
GRAD_ACCUM_STEPS = 2         # Effective batch = 2 GPUs * 2 * 2 = 8.
EPOCHS = 1                  # Full-corpus goal, across as many Kaggle sessions as required.
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 50
LORA_R = 16
LORA_ALPHA = 16
FINETUNE_VISION_LAYERS = True   # Unfreeze vision LoRA from step 107,744 onward while keeping 107.7k language LoRA weights.
GRADIENT_CHECKPOINTING = False # Faster, but enable if batch raises CUDA OOM.
MAX_IMAGE_SIDE = 768          # Speed-first; 896/1024 for tiny Khmer strokes after CER checks.
MAX_SEQ_LENGTH = 2048
MAX_GRAD_NORM = 1.0

# Bounded streaming; no giant TAR extraction and no snapshot_download.
USE_EXISTING_FULL_LOCAL_MIRROR = False
FULL_LOCAL_MIRROR = Path('/tmp/openkhmer_hf_full')
CACHE_ACTIVE_TAR_TO_TMP = True  # Stage active TAR in /tmp SSD to prevent HTTP streaming stalls on GPU.
MAX_SINGLE_TAR_GB = 8
PREFETCH_IMAGES_PER_GPU = 48  # 48 decoded PIL records buffered per GPU worker so GPUs never wait on CPU decode.
SCRATCH_DIR = Path('/tmp/openkhmer_v7_active_tar')

# Session management. First run 100 steps is a *benchmark only*.
SESSION_STEPS = 0           # After benchmark and checkpoint: change to 0 to continue.
MAX_SESSION_HOURS = 6.25      # Keep margin for Kaggle sessions (adjust to actual availability).
STOP_RESERVE_SECONDS = 300   # Save several minutes before planned session boundary.
SAVE_EVERY_STEPS = 500
LOG_EVERY_STEPS = 20

# Critical for repeating this across Kaggle sessions. Optional private remote checkpoint.
PUSH_CHECKPOINTS_TO_HF = True
RESTORE_CHECKPOINT_FROM_HF = True
REMOTE_SYNC_EVERY_N_SAVES = 2  # Local save every 500 steps, HF upload every 1000 steps + always at session end.
CHECKPOINT_REPO = 'dvdphobia/OpenKhmer-Private-Checkpoints'
RESUME_CHECKPOINT_PATH = None  # Option: previous Kaggle output checkpoint directory.

WORK_DIR = Path('/kaggle/working')
INDEX_DIR = WORK_DIR/'openkhmer_v7_full_index'
PLAN_DIR = WORK_DIR/'openkhmer_v7_ddp_plan'
for d in [WORK_DIR,INDEX_DIR,PLAN_DIR,SCRATCH_DIR]: d.mkdir(parents=True,exist_ok=True)

def kaggle_secret(name):
    value=os.environ.get(name)
    if not value:
        try:
            from kaggle_secrets import UserSecretsClient
            value=UserSecretsClient().get_secret(name)
        except Exception: pass
    return value

HF_TOKEN=kaggle_secret('HF_TOKEN')
HF_WRITE_TOKEN=kaggle_secret('HF_WRITE_TOKEN')
if not HF_TOKEN:
    raise RuntimeError('Add Kaggle Secret HF_TOKEN with access to private OpenKhmer repo')
if PUSH_CHECKPOINTS_TO_HF and not HF_WRITE_TOKEN:
    raise RuntimeError('HF_WRITE_TOKEN required to persist private checkpoints')

print('Hugging Face:',HF_DATASET_REPO)
print('Full training (no sample cap):',INCLUDE_MATH_RECOGNITION)
print('Streaming direct from HF:',not USE_EXISTING_FULL_LOCAL_MIRROR and not CACHE_ACTIVE_TAR_TO_TMP)
print('Kaggle /tmp free GiB:',round(shutil.disk_usage('/tmp').free/2**30,2))

## 3 — Verify both Kaggle T4 GPUs **before** starting

The notebook will fail fast if Kaggle gives only one GPU; running two worker processes on one GPU would not be dual-GPU training. Setting `REQUIRE_DUAL_T4=False` enables a one-GPU diagnostic only.


In [ ]:
import torch, subprocess
print(subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True).stdout)
if not torch.cuda.is_available():
    raise RuntimeError('Enable GPU accelerator in Kaggle settings')
GPU_COUNT=torch.cuda.device_count()
if REQUIRE_DUAL_T4 and GPU_COUNT != 2:
    raise RuntimeError(f'Two GPUs REQUIRED. Kaggle currently exposes {GPU_COUNT} GPU(s).')
WORLD_SIZE=2 if REQUIRE_DUAL_T4 else min(2,GPU_COUNT)
for gpu in range(GPU_COUNT):
    p=torch.cuda.get_device_properties(gpu)
    print(f'GPU {gpu}: {p.name}  memory={p.total_memory/2**30:.1f} GiB  capability={p.major}.{p.minor}')
print('DDP workers to launch:',WORLD_SIZE)
assert WORLD_SIZE==2 or not REQUIRE_DUAL_T4


## 4 — Write the self-contained full-streaming reader + dual-GPU runner

This notebook embeds its own Python helpers; you do not need to download additional `.py` files. The indexer streams **only text lists**, checks that every referenced recognition/math train archive exists, and writes per-archive label indexes. The GPU workers use a single sequential pass through different TAR members. Images stay as bounded in-memory PIL records.


In [ ]:
from pathlib import Path
import py_compile
OPENKHMER_STREAM_CORE = r'''"""OpenKhmer remote-streaming or full local mirror TAR training stream.

Remote source layout: recognition/<source>/train_list.txt and
recognition/<source>/image_archives/train.<shard-id>.tar.  No source images
are extracted; a single TAR is optionally cached on disk at a time.
"""
from __future__ import annotations

import hashlib
import io
import json
import math
import os
import random
import shutil
import tarfile
import time
from collections import OrderedDict, defaultdict
from pathlib import Path, PurePosixPath

EXPECTED_EXPORT_TARS = 630  # OpenKhmer 2026-10-21 complete_with_quarantine export
SCHEMA_VERSION = 4


def clean_member_path(value):
    value = str(value).replace('\\', '/')
    while value.startswith('./'):
        value = value[2:]
    path = PurePosixPath(value)
    if path.is_absolute() or '..' in path.parts:
        raise ValueError(f'Unsafe image path: {value!r}')
    return path.as_posix()


def shard_id_from_member(member):
    parts = PurePosixPath(clean_member_path(member)).parts
    if len(parts) < 4 or parts[0] != 'train' or parts[1] != 'images':
        raise ValueError(f'Unexpected PaddleOCR train image path: {member!r}')
    shard = parts[2]
    if not shard or not all(c in '0123456789abcdef' for c in shard.lower()):
        raise ValueError(f'Unexpected train shard ID: {shard!r}')
    return shard


def source_from_list(path):
    parts = PurePosixPath(path).parts
    if len(parts) != 3 or parts[-1] != 'train_list.txt' or parts[0] not in ('recognition','math_recognition'):
        raise ValueError(f'Unexpected source list path: {path!r}')
    return '/'.join(parts[:-1])


def selected_source_dirs(remote_names, include_math=True, allowlist=None):
    names = set(remote_names)
    lists = sorted(p for p in names if p.endswith('/train_list.txt'))
    result = []
    for path in lists:
        try:
            source = source_from_list(path)
        except ValueError:
            continue
        task = source.split('/')[0]
        if task == 'math_recognition' and not include_math:
            continue
        if allowlist is not None and source not in allowlist:
            continue
        result.append((source, task))
    return result


def _interleave_shards(shards, seed):
    """Seeded source-aware mix; every selected shard appears once per epoch."""
    grouped = defaultdict(list)
    for shard in shards:
        grouped[shard['source']].append(shard)
    rng = random.Random(seed)
    for items in grouped.values():
        rng.shuffle(items)
    order = []
    previous = None
    while grouped:
        available = [src for src, items in grouped.items() if items]
        if len(available) > 1 and previous in available:
            available.remove(previous)
        # Weight by remaining shards, not by number of small source files.
        selected = rng.choices(available, weights=[len(grouped[src]) for src in available], k=1)[0]
        order.append(grouped[selected].pop())
        if not grouped[selected]:
            del grouped[selected]
        previous = selected
    return order


class _SmallFilePool:
    def __init__(self, max_open=24):
        self.max_open = max_open
        self.opened = OrderedDict()

    def write(self, filename, row):
        filename = str(filename)
        if filename in self.opened:
            handle = self.opened.pop(filename)
        else:
            if len(self.opened) >= self.max_open:
                _, old_handle = self.opened.popitem(last=False)
                old_handle.close()
            Path(filename).parent.mkdir(parents=True, exist_ok=True)
            handle = open(filename, 'a', encoding='utf-8', buffering=256 * 1024)
        self.opened[filename] = handle
        handle.write(row)

    def close(self):
        for handle in self.opened.values():
            handle.close()
        self.opened.clear()


def prepare_index(repo_id, token, revision, index_dir, include_math=True,
                  allowlist=None, require_complete=True,
                  expected_export_tars=EXPECTED_EXPORT_TARS, seed=3407):
    """Fetch only text label lists and remote file metadata; never downloads TARs."""
    from huggingface_hub import HfApi, HfFileSystem

    index_dir = Path(index_dir)
    index_dir.mkdir(parents=True, exist_ok=True)
    api = HfApi(token=token)
    # Must pin a single immutable dataset revision for all training sessions.
    info = api.repo_info(repo_id=repo_id, repo_type='dataset', revision=revision,
                         files_metadata=True)
    sha = info.sha
    siblings = {f.rfilename: f for f in info.siblings}
    remote_tars = {p: f for p, f in siblings.items() if p.endswith('.tar')}
    total_tars = len(remote_tars)
    if require_complete and total_tars < expected_export_tars:
        raise RuntimeError(
            f'Upload not complete at {sha}: {total_tars}/{expected_export_tars} TARs. '
            'Finish Hugging Face upload, then run the notebook again. '
            'Do not train against the incomplete commit.'
        )

    sources = selected_source_dirs(siblings, include_math=include_math, allowlist=allowlist)
    if not sources:
        raise RuntimeError('No eligible recognition or math train_list.txt found at pinned revision')

    # The whole source index is text-only and is typically far smaller than images.
    # Build it by streaming one label list at a time into per-TAR TSV files.
    # Staging directory is atomic: interrupted indexing is rebuilt, not reused.
    plan = {
        'schema': SCHEMA_VERSION, 'repo_id': repo_id, 'revision': sha,
        'include_math': bool(include_math), 'sources': [s for s, _ in sources],
        'seed': seed, 'expected_export_tars': expected_export_tars,
        'total_repo_tars': total_tars,
    }
    digest = hashlib.sha256(json.dumps(plan, sort_keys=True).encode()).hexdigest()
    summary_path = index_dir / 'corpus_manifest.json'
    if summary_path.exists():
        old = json.loads(summary_path.read_text('utf-8'))
        if old.get('index_fingerprint') == digest and all(
            (index_dir / entry['label_file']).is_file() for entry in old.get('shards', [])
        ):
            return old
    # Remove only files owned by this indexing job, NOT unrelated Kaggle files.
    label_dir = index_dir / 'labels'
    if label_dir.exists():
        shutil.rmtree(label_dir)
    label_dir.mkdir(parents=True, exist_ok=True)
    summary_path.unlink(missing_ok=True)

    fs = HfFileSystem(token=token)
    shards = {}
    source_rows = {}
    pool = _SmallFilePool()
    try:
        for source, task in sources:
            list_path = source + '/train_list.txt'
            source_tar_prefix = source + '/image_archives/train.'
            available = {p for p in remote_tars if p.startswith(source_tar_prefix)}
            if not available:
                raise RuntimeError(f'{source}: missing remote train TARs')
            source_rows[source] = 0
            local_root = os.environ.get('OPENKHMER_LOCAL_REPO_DIR')
            if local_root:
                local_list = Path(local_root) / list_path
                if not local_list.is_file():
                    raise RuntimeError(f'Local mirror missing label list: {local_list}')
                source_stream = open(local_list, 'rt', encoding='utf-8')
            else:
                source_stream = fs.open(f'datasets/{repo_id}@{sha}/{list_path}', 'rt', encoding='utf-8')
            with source_stream as rows:
                for lineno, line in enumerate(rows, 1):
                    row = line.rstrip('\r\n')
                    if not row:
                        continue
                    if '\t' not in row:
                        raise RuntimeError(f'{list_path}:{lineno}: expected TAB-separated image and label')
                    member, label = row.split('\t', 1)
                    member = clean_member_path(member)
                    if not label:
                        raise RuntimeError(f'{list_path}:{lineno}: empty image label')
                    shard_id = shard_id_from_member(member)
                    tar_path = f'{source}/image_archives/train.{shard_id}.tar'
                    if tar_path not in available:
                        raise RuntimeError(f'{list_path}:{lineno}: missing TAR {tar_path}')
                    if tar_path not in shards:
                        relative = f'labels/{hashlib.sha256(tar_path.encode()).hexdigest()[:24]}.tsv'
                        shards[tar_path] = {
                            'remote_tar': tar_path,
                            'source': source,
                            'task': task,
                            'label_file': relative,
                            'num_rows': 0,
                            'tar_bytes': remote_tars[tar_path].size,
                        }
                    pool.write(index_dir / shards[tar_path]['label_file'], member + '\t' + label + '\n')
                    shards[tar_path]['num_rows'] += 1
                    source_rows[source] += 1
            unused = sorted(available - set(shards))
            if unused:
                raise RuntimeError(f'{source}: train TARs without labels: {unused[:3]}')
            print(f'Indexed {source}: {source_rows[source]:,} train labels / {len(available)} shards', flush=True)
    finally:
        pool.close()

    shard_order = _interleave_shards(list(shards.values()), seed=seed)
    total_rows = sum(s['num_rows'] for s in shard_order)
    if total_rows <= 0:
        raise RuntimeError('Empty training corpus')
    result = {**plan, 'index_fingerprint': digest, 'total_rows': total_rows,
              'train_shards': len(shard_order), 'source_rows': source_rows,
              'shards': shard_order}
    temp = summary_path.with_suffix('.json.tmp')
    temp.write_text(json.dumps(result, ensure_ascii=False, indent=2), 'utf-8')
    os.replace(temp, summary_path)
    return result


def _labels_for_shard(label_file):
    labels = {}
    with open(label_file, encoding='utf-8') as rows:
        for lineno, line in enumerate(rows, 1):
            raw = line.rstrip('\r\n')
            member, sep, label = raw.partition('\t')
            if not sep or not label:
                raise RuntimeError(f'Bad shard label row {label_file}:{lineno}')
            if member in labels:
                raise RuntimeError(f'Duplicate image label for {member} in {label_file}')
            labels[member] = label
    return labels


def _stage_tar(fs, repo_id, revision, shard, scratch_dir, min_free_bytes=3_000_000_000,
               max_tar_bytes=8_000_000_000):
    """Single TAR download to bounded local scratch.  Do not use HF hub cache."""
    scratch_dir = Path(scratch_dir)
    scratch_dir.mkdir(parents=True, exist_ok=True)
    path = shard['remote_tar']
    size = shard.get('tar_bytes')
    if size is not None and size > max_tar_bytes:
        raise RuntimeError(f'Archive {path} is too large for cache budget: {size:,} bytes')
    if size is not None and shutil.disk_usage(scratch_dir).free < size + min_free_bytes:
        raise RuntimeError(f'Insufficient Kaggle scratch free space for {path}')

    # Distinct cached name per pinned revision; names cannot escape scratch dir.
    stem = hashlib.sha256((revision + '\0' + path).encode()).hexdigest()[:32]
    dest = scratch_dir / f'{stem}.tar'
    if dest.exists() and (size is None or dest.stat().st_size == size):
        return dest
    # Two DDP processes may request the same shard simultaneously.
    # Use a Linux file lock so they cannot corrupt each other's partial file.
    import fcntl
    lock_path = scratch_dir / f'{stem}.lock'
    with open(lock_path, 'a+b') as lock:
        fcntl.flock(lock, fcntl.LOCK_EX)
        try:
            if dest.exists() and (size is None or dest.stat().st_size == size):
                return dest
            dest.unlink(missing_ok=True)
            temp = dest.with_suffix('.tar.part')
            temp.unlink(missing_ok=True)
            remote_path = f'datasets/{repo_id}@{revision}/{path}'
            try:
                with fs.open(remote_path, 'rb', block_size=8 * 1024 * 1024) as src, open(temp, 'wb') as out:
                    shutil.copyfileobj(src, out, length=8 * 1024 * 1024)
                if size is not None and temp.stat().st_size != size:
                    raise RuntimeError(f'Incomplete TAR transfer {path}: {temp.stat().st_size} != {size}')
                os.replace(temp, dest)
            finally:
                temp.unlink(missing_ok=True)
        finally:
            fcntl.flock(lock, fcntl.LOCK_UN)
    return dest


def iter_training_records(manifest_path, token, start_record=0, repeat_epochs=1,
                          scratch_dir='/kaggle/tmp/openkhmer_shard',
                          index_root=None, use_disk_cache=True,
                          max_tar_gb=8, seed=None, limit_records=None,
                          max_image_side=0):
    """Resume by skipping whole shards in metadata, not downloading old image TARs.

    Emits raw PIL images in TRL prompt/completion format.  Never mixes held-out
    validation, test, evaluation_only, detection, or non_ocr_inputs data.
    """
    from huggingface_hub import HfFileSystem
    from PIL import Image

    manifest_path = Path(manifest_path)
    manifest = json.loads(manifest_path.read_text('utf-8'))
    if manifest['schema'] != SCHEMA_VERSION:
        raise RuntimeError('Corpus manifest schema mismatch')
    if index_root is None:
        index_root = manifest_path.parent
    index_root = Path(index_root)
    n_per_epoch = manifest['total_rows']
    total = n_per_epoch * repeat_epochs
    if not (0 <= start_record < total):
        raise ValueError(f'Invalid resume row {start_record} (total={total})')

    fs = HfFileSystem(token=token)
    done = 0
    for epoch in range(repeat_epochs):
        order = manifest['shards']
        # Keep the same deterministic order for all ranks/sessions.
        for shard in order:
            count = shard['num_rows']
            if done + count <= start_record:
                done += count
                continue
            skip_in_shard = max(0, start_record - done)
            done += count
            label_file = index_root / shard['label_file']
            labels = _labels_for_shard(label_file)
            if len(labels) != count:
                raise RuntimeError(f"Corrupt label index for {shard['remote_tar']}")
            task_prompt = 'Formula Recognition:' if shard['task'] == 'math_recognition' else 'OCR:'
            archive_path = None
            staged = False
            try:
                local_root = os.environ.get('OPENKHMER_LOCAL_REPO_DIR')
                if local_root:
                    archive_path = Path(local_root) / shard['remote_tar']
                    if not archive_path.is_file():
                        raise RuntimeError(f'Local dataset mirror missing TAR: {archive_path}')
                    expect = shard.get('tar_bytes')
                    if expect is not None and archive_path.stat().st_size != expect:
                        raise RuntimeError(f'Local TAR size mismatch: {archive_path}')
                    # Do not remove the local mirror when a shard is consumed.
                    stream = open(archive_path, 'rb')
                elif use_disk_cache:
                    archive_path = _stage_tar(fs, manifest['repo_id'], manifest['revision'], shard,
                                             scratch_dir, max_tar_bytes=int(max_tar_gb * 1e9))
                    staged = True
                    stream = open(archive_path, 'rb')
                else:
                    stream = fs.open(
                        f"datasets/{manifest['repo_id']}@{manifest['revision']}/{shard['remote_tar']}", 'rb',
                        block_size=8 * 1024 * 1024)
                with stream as src, tarfile.open(fileobj=src, mode='r|') as archive:
                    seen = 0
                    for member in archive:
                        if not member.isfile():
                            continue
                        name = clean_member_path(member.name)
                        if name not in labels:
                            continue
                        seen += 1
                        if seen <= skip_in_shard:
                            continue
                        data_stream = archive.extractfile(member)
                        if data_stream is None:
                            raise RuntimeError(f"TAR member unreadable: {shard['remote_tar']}::{name}")
                        data = data_stream.read()
                        try:
                            with Image.open(io.BytesIO(data)) as img:
                                image = img.convert('RGB')
                                if max_image_side and max(image.size) > max_image_side:
                                    image.thumbnail((max_image_side,max_image_side),Image.Resampling.BICUBIC)
                                image.load()
                        except Exception as exc:
                            raise RuntimeError(f"Image decode failed: {shard['remote_tar']}::{name}") from exc
                        yield {
                            'prompt': [{'role':'user','content': [
                                {'type': 'text', 'text': task_prompt},
                                {'type': 'image', 'image': image},
                            ]}],
                            'completion': [{'role':'assistant','content': [
                                {'type':'text', 'text': labels[name]}
                            ]}],
                        }
                        if limit_records is not None:
                            limit_records -= 1
                            if limit_records <= 0:
                                return
                    if seen != count:
                        raise RuntimeError(
                            f"Image/list mismatch in {shard['remote_tar']}: {seen:,}/{count:,} matched"
                        )
            finally:
                # POSIX permits a worker holding an open TAR fd to finish after unlink.
                # Other DDP worker may occasionally redownload it; storage stays bounded.
                if staged and archive_path is not None:
                    archive_path.unlink(missing_ok=True)


def calc_training_steps(total_rows, epochs, batch_per_gpu, accum, world_size):
    if min(total_rows, epochs, batch_per_gpu, accum, world_size) <= 0:
        raise ValueError('All training dimensions must be positive')
    effective = batch_per_gpu * accum * world_size
    return math.ceil(total_rows * epochs / effective)


def resume_offset(global_step, batch_per_gpu, accum, world_size):
    if global_step < 0:
        raise ValueError('global_step must be nonnegative')
    return global_step * batch_per_gpu * accum * world_size


def validate_resume(checkpoint, run_fingerprint):
    checkpoint = Path(checkpoint)
    stamp = checkpoint / 'openkhmer_run.json'
    if not stamp.is_file():
        raise RuntimeError(f'Refusing unverified checkpoint: {stamp} absent')
    existing = json.loads(stamp.read_text('utf-8'))
    if existing.get('run_fingerprint') != run_fingerprint:
        raise RuntimeError('Checkpoint fingerprint mismatch: dataset/model/batch/config changed')
    state_path = checkpoint / 'trainer_state.json'
    if not state_path.is_file():
        raise RuntimeError('Checkpoint missing trainer_state.json')
    state = json.loads(state_path.read_text('utf-8'))
    if not (checkpoint / 'optimizer.pt').exists() and not (checkpoint / 'optimizer.bin').exists():
        raise RuntimeError('Checkpoint missing optimizer state: cannot resume training exactly')
    if not (checkpoint / 'scheduler.pt').exists():
        raise RuntimeError('Checkpoint missing scheduler state')
    return int(state['global_step'])


def make_run_fingerprint(dataset_manifest, model_id, include_math, epochs,
                         batch_per_gpu, accum, world_size, lora_config, preprocess=None):
    details = {
        'revision':dataset_manifest['revision'],
        'corpus_index_fingerprint':dataset_manifest['index_fingerprint'],
        'total_rows':dataset_manifest['total_rows'],
        'model_id': model_id,
        'include_math': bool(include_math),
        'epochs':epochs,
        'batch_per_gpu':batch_per_gpu,
        'accum':accum,
        'world_size':world_size,
        'lora_config':lora_config,
        'preprocess':preprocess or {},
    }
    return hashlib.sha256(json.dumps(details, sort_keys=True).encode()).hexdigest()


def streaming_generator(manifest_path, start_record, repeat_epochs,
                        scratch_dir, use_disk_cache, max_tar_gb, max_image_side=0):
    # Token stays in the worker environment, never inside serialized dataset metadata.
    token = os.environ.get('OPENKHMER_HF_TOKEN')
    return iter_training_records(
        manifest_path=manifest_path, token=token,
        start_record=start_record, repeat_epochs=repeat_epochs,
        scratch_dir=scratch_dir, use_disk_cache=use_disk_cache,
        max_tar_gb=max_tar_gb, max_image_side=max_image_side)

class OpenKhmerTorchDataset:
    """Factory for a torch IterableDataset without any Arrow/PIL reencoding."""
    @staticmethod
    def create(manifest_path, start_record, repeat_epochs,
               scratch_dir, use_disk_cache, max_tar_gb, max_image_side=0):
        from torch.utils.data import IterableDataset

        class _Dataset(IterableDataset):
            def __iter__(self):
                yield from streaming_generator(
                    manifest_path=manifest_path,
                    start_record=start_record,
                    repeat_epochs=repeat_epochs,
                    scratch_dir=scratch_dir,
                    use_disk_cache=use_disk_cache,
                    max_tar_gb=max_tar_gb,
                    max_image_side=max_image_side,
                )
        return _Dataset()
'''
OPENKHMER_DDP_PLAN = r'''"""Full-corpus two-rank TAR-shard schedule and bounded prefetch for OpenKhmer.

This module deliberately has no HF or PyTorch dependencies, so its scheduling,
quota handling and resume semantics can be tested offline.
"""
from __future__ import annotations

import hashlib
import json
import math
import os
from pathlib import Path
from queue import Queue
from threading import Thread


PLAN_SCHEMA = 1


def atomic_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(data, ensure_ascii=False, indent=2) + '\n', 'utf-8')
    os.replace(tmp, path)


def partition_full_corpus(corpus, out_dir, world_size=2, batch_per_gpu=1, accumulation=4, epochs=1):
    """Assign every TAR to exactly one GPU; pad only the shorter rank by replay.

    LPT scheduling balances image counts; original interleaved order is restored
    inside each rank to preserve source mixing. No archive is assigned twice.
    """
    if not (1 <= world_size <= 8 and batch_per_gpu >= 1 and accumulation >= 1 and epochs >= 1):
        raise ValueError('Invalid distributed training dimensions')
    if not corpus.get('shards'):
        raise ValueError('Empty corpus shard list')
    indexed = list(enumerate(corpus['shards']))
    assignments = [[] for _ in range(world_size)]
    counts = [0] * world_size
    for index, shard in sorted(indexed, key=lambda item: (-item[1]['num_rows'], item[0])):
        rank = min(range(world_size), key=lambda r: (counts[r], r))
        assignments[rank].append((index, shard))
        counts[rank] += int(shard['num_rows'])

    if min(counts) <= 0:
        raise ValueError('Need at least one training TAR per training GPU')
    if sum(counts) != corpus['total_rows']:
        raise RuntimeError('Shard counts disagree with original corpus')
    by_gpu = batch_per_gpu * accumulation
    target_steps = math.ceil(max(counts) * epochs / by_gpu)
    per_gpu_appearances = target_steps * by_gpu
    all_shards = [s['remote_tar'] for rank in assignments for _, s in rank]
    if len(all_shards) != len(set(all_shards)):
        raise RuntimeError('A TAR was assigned to multiple GPUs')
    if set(all_shards) != {s['remote_tar'] for s in corpus['shards']}:
        raise RuntimeError('TAR schedule omitted at least one source')

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for rank, rank_shards in enumerate(assignments):
        ordered = [shard for _, shard in sorted(rank_shards, key=lambda item: item[0])]
        rank_manifest = dict(corpus)
        rank_manifest.update({
            'total_rows': counts[rank],
            'shards': ordered,
            'rank': rank,
            'world_size': world_size,
            'parent_index_fingerprint': corpus['index_fingerprint'],
        })
        output = out_dir / f'rank-{rank}-corpus.json'
        atomic_json(output, rank_manifest)
        paths.append(str(output))

    result = {
        'schema': PLAN_SCHEMA,
        'repository_sha': corpus['revision'],
        'parent_index_fingerprint': corpus['index_fingerprint'],
        'world_size': world_size,
        'batch_per_gpu': batch_per_gpu,
        'gradient_accumulation': accumulation,
        'epochs': epochs,
        'unique_training_records': corpus['total_rows'] * epochs,
        'per_rank_unique_records': counts,
        'per_rank_training_presentations': per_gpu_appearances,
        'padded_presentations_per_rank': [per_gpu_appearances - n * epochs for n in counts],
        'total_optimizer_steps': target_steps,
        'effective_batch_size': world_size * by_gpu,
        'rank_manifests': paths,
    }
    # Config contents, never local absolute paths, define reproducible training fingerprint.
    identifying = {k: v for k, v in result.items() if k != 'rank_manifests'}
    result['plan_fingerprint'] = hashlib.sha256(
        json.dumps(identifying, sort_keys=True).encode('utf-8')
    ).hexdigest()
    atomic_json(out_dir / 'ddp_plan.json', result)
    return result


def record_stream_for_rank(base_iter_fn, rank_manifest, index_dir, *, start_row, target_rows):
    """Position-based resumable iterator: repeats ONLY at epoch/padding boundary.

    `base_iter_fn` is provided by the existing verified OpenKhmer TAR reader.
    The caller stores a single optimizer-step cursor and thus resumes by row.
    """
    if start_row < 0 or target_rows < 0 or start_row > target_rows:
        raise ValueError('Invalid stream boundaries')
    rank_rows = rank_manifest['total_rows']
    if rank_rows <= 0:
        raise ValueError('Empty rank corpus')
    remaining = target_rows - start_row
    current = start_row % rank_rows
    while remaining:
        observed = 0
        for example in base_iter_fn(start_record=current):
            yield example
            remaining -= 1
            observed += 1
            if not remaining:
                return
        if observed != rank_rows - current:
            raise RuntimeError(f'Incomplete rank TAR stream: {observed} != {rank_rows-current}')
        current = 0


class BoundedPrefetch:
    """Overlap image IO/decode with GPU work, with bounded CPU memory."""
    def __init__(self, iterable, maxsize=8):
        if maxsize < 1:
            raise ValueError('maxsize must be positive')
        self.iterator = iter(iterable)
        self.queue = Queue(maxsize=maxsize)
        self.thread = Thread(target=self._fill, daemon=True)
        self.thread.start()

    def _fill(self):
        try:
            for value in self.iterator:
                self.queue.put((0, value))
        except BaseException as err:
            self.queue.put((1, err))
        finally:
            self.queue.put((2, None))

    def __iter__(self):
        while True:
            tag, value = self.queue.get()
            if tag == 0:
                yield value
            elif tag == 1:
                raise value
            else:
                break
'''
OPENKHMER_DDP_RUNNER = r'''"""OpenKhmer v7 true 2-T4 DDP, full eligible OCR-corpus, resumable TAR streaming.

Launched by torchrun from the self-contained Kaggle notebook. Each worker owns
its own disjoint assigned TAR shards, so both GPUs decode different images.
"""
from __future__ import annotations

import contextlib
import hashlib
import itertools
import json
import math
import os
import shutil
import sys
import io
import queue
import random
import tarfile
import threading
import time
from pathlib import Path
from PIL import Image

# IMPORTANT: Unsloth import must precede Transformers / PEFT.
from unsloth import FastVisionModel
import torch
import torch.distributed as dist
from peft import PeftModel
from transformers import AutoProcessor
from unsloth.trainer import UnslothVisionDataCollator
from openkhmer_stream_core import iter_training_records
from openkhmer_ddp_plan import BoundedPrefetch, record_stream_for_rank


CONFIG = json.loads(Path(os.environ['OPENKHMER_V7_RUN_CONFIG']).read_text('utf-8'))
RANK = int(os.environ.get('RANK', '0'))
LOCAL_RANK = int(os.environ.get('LOCAL_RANK', '0'))
WORLD = int(os.environ.get('WORLD_SIZE', '1'))
if WORLD != int(CONFIG['world_size']):
    raise RuntimeError(f'World size mismatch: {WORLD} != {CONFIG["world_size"]}')
if not torch.cuda.is_available() or torch.cuda.device_count() < WORLD:
    raise RuntimeError('The requested number of Kaggle GPUs is not available')
torch.cuda.set_device(LOCAL_RANK)
DEVICE = torch.device('cuda', LOCAL_RANK)
if WORLD > 1:
    dist.init_process_group(backend='nccl', timeout=__import__('datetime').timedelta(minutes=30))
torch.manual_seed(int(CONFIG['seed']))
torch.cuda.manual_seed_all(int(CONFIG['seed']))


def info(*args):
    if RANK == 0:
        print(*args, flush=True)


def barrier():
    if WORLD > 1:
        dist.barrier()


def sync_stop(flag):
    tensor = torch.tensor([int(bool(flag)) if RANK == 0 else 0], device=DEVICE)
    if WORLD > 1:
        dist.broadcast(tensor, src=0)
    return bool(tensor.item())


PLAN = json.loads(Path(CONFIG['plan_path']).read_text('utf-8'))
RANK_MANIFEST_PATH = Path(PLAN['rank_manifests'][RANK])
RANK_MANIFEST = json.loads(RANK_MANIFEST_PATH.read_text('utf-8'))
TOTAL_STEPS = PLAN['total_optimizer_steps']
LOCAL_BATCH = int(CONFIG['batch_per_gpu'])
ACCUM = int(CONFIG['gradient_accumulation'])
LOCAL_PER_STEP = LOCAL_BATCH * ACCUM
if PLAN['repository_sha'] != CONFIG['dataset_revision']:
    raise RuntimeError('Dataset repository revision changed')
if RANK_MANIFEST['rank'] != RANK or RANK_MANIFEST['world_size'] != WORLD:
    raise RuntimeError('Wrong per-rank shard assignment')
if not torch.cuda.is_available():
    raise RuntimeError('Enable Kaggle GPU')

local_repo = CONFIG.get('local_repo_dir')
if local_repo:
    mirror = Path(local_repo)
    state_file = mirror / '.openkhmer_mirror_state.json'
    if not state_file.is_file():
        raise RuntimeError('Unverified full /tmp mirror; disable USE_FULL_LOCAL_MIRROR')
    mirror_state = json.loads(state_file.read_text('utf-8'))
    if mirror_state.get('revision') != CONFIG['dataset_revision'] or not mirror_state.get('complete_export_tars'):
        raise RuntimeError('Full local /tmp mirror is incomplete or its HF SHA changed')
    os.environ['OPENKHMER_LOCAL_REPO_DIR'] = str(mirror)
else:
    os.environ.pop('OPENKHMER_LOCAL_REPO_DIR', None)

CHECKPOINT_DIR = Path(CONFIG['checkpoint_dir'])
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESUME = CONFIG.get('resume_path')
step_start = 0
if RESUME:
    meta = json.loads((Path(RESUME) / 'state.json').read_text('utf-8'))
    if meta.get('run_fingerprint') != CONFIG['run_fingerprint']:
        raise RuntimeError('Checkpoint mismatch: revision, training parameters or source index changed')
    if meta.get('world_size') != WORLD:
        raise RuntimeError('Checkpoint GPU configuration changed')
    step_start = int(meta['optimizer_step'])
    if not all((Path(RESUME)/x).exists() for x in ('adapter','optimizer.pt','scheduler.pt','scaler.pt')):
        raise RuntimeError('Incomplete resume checkpoint')
if step_start >= TOTAL_STEPS:
    raise RuntimeError('Complete run: all planned training steps already finished')

info('OpenKhmer v7 FULL CORPUS TRUE DDP')
info('World GPUs:', WORLD, '| model:', CONFIG['model_id'])
info('HF revision:', CONFIG['dataset_revision'])
info('Training unique rows:', f'{PLAN["unique_training_records"]:,}')
info('Rank rows:', PLAN['per_rank_unique_records'])
info('Padded presentations:', PLAN['padded_presentations_per_rank'])
info('Per-rank archive counts:', [len(json.loads(Path(p).read_text('utf-8'))['shards']) for p in PLAN['rank_manifests']])
info('Effective batch:', WORLD * LOCAL_BATCH * ACCUM, '| planned steps:', f'{TOTAL_STEPS:,}')
info('Resume optimizer step:', step_start)
print(f'Rank {RANK} maps to GPU {LOCAL_RANK}: {torch.cuda.get_device_name(LOCAL_RANK)}', flush=True)

# Reject recursively patched rope methods from old notebooks.
import transformers.models.paddleocr_vl.modeling_paddleocr_vl as paddle_module
if getattr(paddle_module.PaddleOCRVLModel.compute_3d_position_ids,
           '__name__', '') == 'safe_compute_3d_position_ids':
    raise RuntimeError('Legacy recursive M-RoPE monkeypatch detected; restart Kaggle kernel')

base, _ = FastVisionModel.from_pretrained(
    model_name=CONFIG['model_id'],
    max_seq_length=int(CONFIG['max_seq_length']),
    load_in_4bit=True,
    device_map={'': LOCAL_RANK},
)
processor = AutoProcessor.from_pretrained(CONFIG['model_id'], trust_remote_code=True)
upgraded_vision_lora = False
if RESUME:
    saved_cfg = json.loads((Path(RESUME)/'adapter'/'adapter_config.json').read_text('utf-8'))
    has_vision = 'vision' in str(saved_cfg.get('target_modules', ''))
    if bool(CONFIG['finetune_vision_layers']) and not has_vision:
        info('Upgrading step', step_start, 'language-only LoRA -> Vision + Language LoRA (preserving all 252 language LoRA tensors)!')
        model = FastVisionModel.get_peft_model(
            base,
            finetune_vision_layers=True,
            finetune_language_layers=True,
            finetune_attention_modules=True,
            finetune_mlp_modules=True,
            r=int(CONFIG['lora_r']),
            lora_alpha=int(CONFIG['lora_alpha']),
            lora_dropout=0,
            bias='none',
            random_state=int(CONFIG['seed']),
        )
        from safetensors.torch import load_file
        from peft import set_peft_model_state_dict
        old_weights = load_file(str(Path(RESUME)/'adapter'/'adapter_model.safetensors'), device='cpu')
        set_peft_model_state_dict(model, old_weights)
        del old_weights
        upgraded_vision_lora = True
    else:
        model = PeftModel.from_pretrained(base, str(Path(RESUME)/'adapter'), is_trainable=True)
else:
    model = FastVisionModel.get_peft_model(
        base,
        finetune_vision_layers=bool(CONFIG['finetune_vision_layers']),
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
        r=int(CONFIG['lora_r']),
        lora_alpha=int(CONFIG['lora_alpha']),
        lora_dropout=0,
        bias='none',
        random_state=int(CONFIG['seed']),
    )
FastVisionModel.for_training(model)
if hasattr(model.config, 'use_cache'):
    model.config.use_cache = False
if CONFIG['gradient_checkpointing']:
    if not hasattr(model, 'gradient_checkpointing_enable'):
        raise RuntimeError('Model has no gradient_checkpointing_enable method')
    try:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})
    except TypeError:
        model.gradient_checkpointing_enable()

# Quantized base model must remain entirely on its own local CUDA device.
cuda_owners = {p.device.index for p in model.parameters() if p.device.type == 'cuda'}
if cuda_owners != {LOCAL_RANK}:
    raise RuntimeError(f'Rank {RANK} model on wrong GPUs: {cuda_owners}')
trainable = [p for p in model.parameters() if p.requires_grad]
if not trainable:
    raise RuntimeError('LoRA produced zero trainable parameters')

if WORLD == 2:
    ddp = torch.nn.parallel.DistributedDataParallel(
        model,
        device_ids=[LOCAL_RANK],
        output_device=LOCAL_RANK,
        broadcast_buffers=False,
        find_unused_parameters=False,
        gradient_as_bucket_view=True,
    )
else:
    ddp = model

# 8-bit paged optimizer minimizes memory; it does not re-quantize the base model.
import bitsandbytes as bnb
optimizer = bnb.optim.PagedAdamW8bit(
    trainable,
    lr=float(CONFIG['learning_rate']),
    weight_decay=float(CONFIG['weight_decay']),
)

warmup = int(CONFIG['warmup_steps'])
def lr_multiplier(step):
    if step < warmup:
        return min(1.0, float(step + 1) / max(1, warmup))
    if step <= 107744:
        return 1.0
    progress = min(1.0, float(step - 107744) / max(1.0, float(TOTAL_STEPS - 107744)))
    base_mult = 0.15 + 0.85 * 0.5 * (1.0 + math.cos(math.pi * progress))
    # After 130k steps, dampen LR by 0.35x (~3.5e-5) so Vision+Language LoRA weights converge smoothly without step-to-step oscillation
    return base_mult * 0.35 if step >= 130000 else base_mult
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_multiplier)
try:
    scaler = torch.amp.GradScaler('cuda')
except TypeError:
    scaler = torch.cuda.amp.GradScaler()
if RESUME:
    if not upgraded_vision_lora:
        optimizer.load_state_dict(torch.load(Path(RESUME)/'optimizer.pt', map_location=DEVICE, weights_only=False))
    else:
        info('Vision LoRA layers added at step', step_start, '-> fresh 8-bit Adam state initialized; next save will persist full Vision+Language optimizer.')
    scheduler.load_state_dict(torch.load(Path(RESUME)/'scheduler.pt', map_location='cpu', weights_only=False))
    scaler.load_state_dict(torch.load(Path(RESUME)/'scaler.pt', map_location='cpu', weights_only=False))

image_token_id = getattr(model.config, 'image_token_id', None)
video_token_id = getattr(model.config, 'video_token_id', None)
if image_token_id is None:
    raise RuntimeError('PaddleOCR-VL image token is missing')
base_collator = UnslothVisionDataCollator(
    model, processor, resize='max', completion_only_loss=True,
)
base_collator.max_seq_length = None


def collate(examples):
    out = base_collator(examples)
    tokens = out['input_ids']
    if 'attention_mask' not in out:
        out['attention_mask'] = torch.ones_like(tokens)
    for key in ('pixel_values', 'image_grid_thw', 'labels'):
        if key not in out:
            raise RuntimeError('Multimodal collator missing ' + key)
    if out.get('mm_token_type_ids') is None:
        mm = torch.zeros_like(tokens, dtype=torch.int32)
        mm[tokens == image_token_id] = 1
        if video_token_id is not None:
            mm[tokens == video_token_id] = 2
        out['mm_token_type_ids'] = mm
    mask = tokens == image_token_id
    if not mask.any().item():
        raise RuntimeError('Image chat template has no image tokens')
    out['labels'][mask] = -100
    out['labels'][out['attention_mask'] == 0] = -100
    if not (out['labels'] != -100).any().item():
        raise RuntimeError('No OCR label tokens supervised in batch')
    return {k: v.to(DEVICE, non_blocking=True) if torch.is_tensor(v) else v
            for k, v in out.items()}


rank_rows = RANK_MANIFEST['total_rows']
start_row = step_start * LOCAL_PER_STEP
local_target = PLAN['per_rank_training_presentations']
if start_row >= local_target:
    raise RuntimeError('No examples remaining at this rank')


def make_base_iterator(start_record):
    return iter_training_records(
        manifest_path=RANK_MANIFEST_PATH,
        token=os.environ.get('OPENKHMER_HF_TOKEN'),
        start_record=start_record,
        repeat_epochs=1,
        scratch_dir=str(Path(CONFIG['scratch_dir']) / f'rank{RANK}'),
        index_root=Path(CONFIG['corpus_index_dir']),
        use_disk_cache=bool(CONFIG['cache_tars_on_disk']),
        max_tar_gb=float(CONFIG['max_single_tar_gb']),
        max_image_side=int(CONFIG['max_image_side']),
    )

samples = record_stream_for_rank(
    make_base_iterator, RANK_MANIFEST, Path(CONFIG['corpus_index_dir']),
    start_row=start_row, target_rows=local_target,
)
if CONFIG['prefetch_queue']:
    samples = BoundedPrefetch(samples, maxsize=int(CONFIG['prefetch_queue']))
data_iter = iter(samples)
# Probe uses the SAME iterator; the first image is returned to actual training.
first = next(data_iter)
first_batch = collate([first])
info('Vision collator first-batch check PASSED; actual TAR records, no fake data.')
del first_batch
iterator = itertools.chain([first], data_iter)


# Background stream for detection/ folders (SoyVitou, KhmerST, Darayut_v2, Darayut)
# Produces:
# 1) Real-world cropped & multi-line scene OCR ('OCR:') from KhmerST & SoyVitou
# 2) Normalized [0..1000] bounding-box detection ('Text Detection:') from all 4 detection datasets
DETECTION_SOURCES = [
    ('detection/SoyVitou_Khmer-Text-Detection-0.2k-d38027b972', 'train.24e9b2544ed73458.tar', True),
    ('detection/KhmerST-fff43542da', 'train.6a04d52ceb50eb5a.tar', True),
    ('detection/Darayut_khmer-textline-dataset_v2-2d3861cd53', 'train.7f16c8bd34e7d8bd.tar', False),
    ('detection/Darayut_Textline-Detection-Dataset-936067bbf8', 'train.17292e220c3284a8.tar', False),
]
det_queue = queue.Queue(maxsize=24)

def _detection_worker():
    from huggingface_hub import HfFileSystem
    from openkhmer_stream_core import clean_member_path
    token = os.environ.get('OPENKHMER_HF_TOKEN')
    repo_id = RANK_MANIFEST['repo_id']
    rev = CONFIG['dataset_revision']
    max_side = int(CONFIG['max_image_side'])
    fs = HfFileSystem(token=token)
    # Offset starting source by RANK so GPU 0 and GPU 1 stream different detection archives
    sources = DETECTION_SOURCES[RANK:] + DETECTION_SOURCES[:RANK]
    while True:
        for src_dir, tar_name, has_transcripts in sources:
            try:
                list_url = f'datasets/{repo_id}@{rev}/{src_dir}/train_list.txt'
                labels = {}
                with fs.open(list_url, 'rt', encoding='utf-8') as fh:
                    for line in fh:
                        row = line.strip()
                        if not row or '\t' not in row:
                            continue
                        m_path, raw_json = row.split('\t', 1)
                        labels[clean_member_path(m_path)] = json.loads(raw_json)
                tar_url = f'datasets/{repo_id}@{rev}/{src_dir}/image_archives/{tar_name}'
                with fs.open(tar_url, 'rb', block_size=8 * 1024 * 1024) as src_stream, tarfile.open(fileobj=src_stream, mode='r|') as archive:
                    for member in archive:
                        if not member.isfile():
                            continue
                        name = clean_member_path(member.name)
                        if name not in labels:
                            continue
                        items = labels[name]
                        if not items:
                            continue
                        fobj = archive.extractfile(member)
                        if fobj is None:
                            continue
                        raw_bytes = fobj.read()
                        with Image.open(io.BytesIO(raw_bytes)) as raw_im:
                            orig_im = raw_im.convert('RGB')
                            orig_w, orig_h = orig_im.size
                            if orig_w < 16 or orig_h < 16:
                                continue
                            # Parse valid boxes [x1, y1, x2, y2] + transcriptions
                            parsed = []
                            for it in items:
                                pts = it.get('points') or []
                                if len(pts) < 4:
                                    continue
                                xs = [int(round(p[0])) for p in pts]
                                ys = [int(round(p[1])) for p in pts]
                                x1, y1 = max(0, min(xs)), max(0, min(ys))
                                x2, y2 = min(orig_w, max(xs)), min(orig_h, max(ys))
                                if x2 - x1 >= 6 and y2 - y1 >= 6:
                                    parsed.append((x1, y1, x2, y2, (it.get('transcription') or '').strip()))
                            if not parsed:
                                continue
                            # Sort top-to-bottom, left-to-right
                            parsed.sort(key=lambda b: (b[1] // max(12, int(orig_h * 0.025)), b[0]))
                            # A) If source has transcripts (KhmerST / SoyVitou), emit cropped line OCR & full-image multi-line OCR
                            if has_transcripts:
                                text_lines = [b[4] for b in parsed if b[4]]
                                for x1, y1, x2, y2, txt in parsed[:6]:
                                    if txt and (x2 - x1) >= 14 and (y2 - y1) >= 10:
                                        crop = orig_im.crop((x1, y1, x2, y2))
                                        if max_side and max(crop.size) > max_side:
                                            crop.thumbnail((max_side, max_side), Image.Resampling.BICUBIC)
                                        det_queue.put({
                                            'prompt': [{'role': 'user', 'content': [{'type': 'text', 'text': 'OCR:'}, {'type': 'image', 'image': crop}]}],
                                            'completion': [{'role': 'assistant', 'content': [{'type': 'text', 'text': txt}]}],
                                        })
                                if len(text_lines) >= 2:
                                    full_ocr_im = orig_im.copy()
                                    if max_side and max(full_ocr_im.size) > max_side:
                                        full_ocr_im.thumbnail((max_side, max_side), Image.Resampling.BICUBIC)
                                    det_queue.put({
                                        'prompt': [{'role': 'user', 'content': [{'type': 'text', 'text': 'OCR:'}, {'type': 'image', 'image': full_ocr_im}]}],
                                        'completion': [{'role': 'assistant', 'content': [{'type': 'text', 'text': '\n'.join(text_lines[:40])}]}],
                                    })
                            # B) Emit normalized [0..1000] bounding-box detection task ('Text Detection:')
                            norm_boxes = [
                                [
                                    int(round(x1 * 1000.0 / orig_w)),
                                    int(round(y1 * 1000.0 / orig_h)),
                                    int(round(x2 * 1000.0 / orig_w)),
                                    int(round(y2 * 1000.0 / orig_h)),
                                ]
                                for x1, y1, x2, y2, _ in parsed[:50]
                            ]
                            det_im = orig_im.copy()
                            if max_side and max(det_im.size) > max_side:
                                det_im.thumbnail((max_side, max_side), Image.Resampling.BICUBIC)
                            det_queue.put({
                                'prompt': [{'role': 'user', 'content': [{'type': 'text', 'text': 'Text Detection:'}, {'type': 'image', 'image': det_im}]}],
                                'completion': [{'role': 'assistant', 'content': [{'type': 'text', 'text': json.dumps(norm_boxes, separators=(',', ':'))}]}],
                            })
            except Exception as det_err:
                info('Detection stream warning on', src_dir, ':', det_err)
                time.sleep(2.0)

threading.Thread(target=_detection_worker, daemon=True).start()
info('Started background detection/ multitask stream (SoyVitou + KhmerST + Darayut v1/v2).')


# Background Multi-Domain Hard-Dataset Stream:
# Ensures Handwritten Khmer (4.2k), Decorative/Moul Fonts (42k), WildKhmerST (23k),
# EmHengly (124k), KiteAether (6.3k), vichetkao (22k), SoyVitou_62k, Sokheng, seanghay,
# LazyGreed, KimkosalYon, and Khmer Math are continually mixed in alongside the giant
# synthetic bulk shards (SoyVitou_1M / mrrtmob) so the Vision Tower never overfits to 1 font!
BULK_SYNTHETIC_SOURCES = {
    'recognition/mrrtmob_khmer_english_ocr_image_line-38c2033c24',
    'recognition/SoyVitou_KhmerSynthetic1M-680e080d2f',
}
diverse_queue = queue.Queue(maxsize=32)

def _diverse_domain_worker():
    from collections import defaultdict
    corpus_manifest_path = Path(CONFIG['corpus_index_dir']) / 'corpus_manifest.json'
    if not corpus_manifest_path.is_file():
        return
    full_corpus = json.loads(corpus_manifest_path.read_text('utf-8'))
    by_source = defaultdict(list)
    for sh in full_corpus.get('shards', []):
        if sh.get('source') not in BULK_SYNTHETIC_SOURCES:
            by_source[sh['source']].append(sh)
    if not by_source:
        return
    rng = random.Random(int(CONFIG['seed']) + 101 + RANK * 37)
    while True:
        # Round-robin across all 12 diverse sources so small gold datasets (handwritten, WildKhmerST, fonts)
        # get equal representation in every cycle!
        round_shards = []
        src_keys = sorted(by_source.keys())
        rng.shuffle(src_keys)
        for sk in src_keys:
            cand_list = by_source[sk]
            round_shards.append(cand_list[rng.randrange(len(cand_list))])
        sub_manifest_path = Path(CONFIG['scratch_dir']) / f'diverse_rank{RANK}.json'
        sub_manifest_path.parent.mkdir(parents=True, exist_ok=True)
        sub_manifest = dict(full_corpus)
        sub_manifest['shards'] = round_shards
        sub_manifest['total_rows'] = sum(int(s['num_rows']) for s in round_shards)
        sub_manifest_path.write_text(json.dumps(sub_manifest), 'utf-8')
        try:
            for rec in iter_training_records(
                manifest_path=sub_manifest_path,
                token=os.environ.get('OPENKHMER_HF_TOKEN'),
                start_record=0,
                repeat_epochs=1,
                scratch_dir=str(Path(CONFIG['scratch_dir']) / f'diverse_tar_rank{RANK}'),
                index_root=Path(CONFIG['corpus_index_dir']),
                use_disk_cache=False,
                max_tar_gb=float(CONFIG['max_single_tar_gb']),
                max_image_side=int(CONFIG['max_image_side']),
            ):
                diverse_queue.put(rec)
        except Exception as div_err:
            info('Diverse domain stream warning:', div_err)
            time.sleep(2.0)

threading.Thread(target=_diverse_domain_worker, daemon=True).start()
info('Started 12-source Multi-Domain Hard-Dataset Mixer (Handwritten + Moul/Fonts + WildKhmerST + EmHengly + Math).')


def _augment_ocr_visuals(img):
    # Applies realistic color ink, tinted backgrounds, small-text scaling, or JPEG compression.
    import numpy as np
    from PIL import ImageFilter
    w, h = img.size
    if w < 24 or h < 16:
        return img
    # 1) 25% chance: recolor black-on-white synthetic text into colored ink / tinted box background
    if random.random() < 0.25:
        arr = np.array(img, dtype=np.float32)
        gray = arr.mean(axis=2, keepdims=True) / 255.0  # 0=ink, 1=bg
        ink_palettes = [
            (180, 20, 30),   # Crimson red (like poem titles/stanzas)
            (20, 60, 175),   # Royal blue
            (20, 125, 55),   # Forest green
            (130, 30, 145),  # Purple
            (140, 85, 20),   # Golden brown
            (35, 45, 65),    # Slate dark ink
        ]
        bg_palettes = [
            (255, 255, 255), # Pure white
            (254, 250, 230), # Warm cream / yellow box
            (238, 246, 255), # Soft blue org-chart box
            (242, 244, 247), # Light gray form cell
        ]
        ink_rgb = np.array(random.choice(ink_palettes), dtype=np.float32).reshape(1, 1, 3)
        bg_rgb = np.array(random.choice(bg_palettes), dtype=np.float32).reshape(1, 1, 3)
        blended = gray * bg_rgb + (1.0 - gray) * ink_rgb
        img = Image.fromarray(np.clip(blended, 0, 255).astype(np.uint8))
    # 2) 10% chance: gentle scale variation (0.72x-0.90x) without destroying crisp subscript strokes
    if h > 32 and w > 32 and random.random() < 0.10:
        scale = random.uniform(0.72, 0.90)
        small = img.resize((max(24, int(w * scale)), max(24, int(h * scale))), Image.Resampling.BICUBIC)
        img = small.resize((w, h), Image.Resampling.BICUBIC)
    # 3) 8% chance: light JPEG compression (quality 72-92)
    if random.random() < 0.08:
        buf = io.BytesIO()
        img.save(buf, format='JPEG', quality=random.randint(72, 92))
        buf.seek(0)
        img = Image.open(buf).convert('RGB')
    return img


def load_batch():
    records = [next(iterator) for _ in range(LOCAL_BATCH)]
    # 50% Multi-Domain Curriculum Mixing in the second slot of each micro-batch:
    # - 30% from the 12 diverse datasets (Handwritten, Moul/Fonts, WildKhmerST, EmHengly, Math, etc.)
    # - 20% from the 4 detection/ folders (KhmerST, SoyVitou, Darayut v1/v2 scene OCR + Text Detection)
    roll = random.random()
    if roll < 0.30:
        try:
            records[-1] = diverse_queue.get_nowait()
        except queue.Empty:
            pass
    elif roll < 0.50:
        try:
            records[-1] = det_queue.get_nowait()
        except queue.Empty:
            pass

    for rec in records:
        p_content = rec['prompt'][0]['content']
        prompt_txt = p_content[0].get('text', '')
        img = p_content[1].get('image')
        if prompt_txt == 'OCR:' and img is not None:
            p_content[1]['image'] = _augment_ocr_visuals(img)

    # 35% chance: stitch 2 single-line OCR records into a multi-line paragraph block ('\n')
    # (Consumes the exact same LOCAL_BATCH records from iterator so resume row offsets never drift!)
    if (
        len(records) == 2
        and records[0]['prompt'][0]['content'][0].get('text') == 'OCR:'
        and records[1]['prompt'][0]['content'][0].get('text') == 'OCR:'
        and '\n' not in records[0]['completion'][0]['content'][0].get('text', '')
        and '\n' not in records[1]['completion'][0]['content'][0].get('text', '')
        and random.random() < 0.35
    ):
        img_a = records[0]['prompt'][0]['content'][1]['image']
        img_b = records[1]['prompt'][0]['content'][1]['image']
        txt_a = records[0]['completion'][0]['content'][0]['text']
        txt_b = records[1]['completion'][0]['content'][0]['text']
        gap = random.randint(6, 18)
        pad = 8
        canvas_w = max(img_a.width, img_b.width) + pad * 2
        canvas = Image.new('RGB', (canvas_w, img_a.height + img_b.height + gap + pad * 2), (255, 255, 255))
        x_a = pad if random.random() < 0.75 else max(pad, (canvas_w - img_a.width) // 2)
        x_b = pad if random.random() < 0.75 else max(pad, (canvas_w - img_b.width) // 2)
        canvas.paste(img_a, (x_a, pad))
        canvas.paste(img_b, (x_b, pad + img_a.height + gap))
        records = [{
            'prompt': [{'role': 'user', 'content': [
                {'type': 'text', 'text': 'OCR:'},
                {'type': 'image', 'image': canvas},
            ]}],
            'completion': [{'role': 'assistant', 'content': [
                {'type': 'text', 'text': f'{txt_a}\n{txt_b}'},
            ]}],
        }]
    return collate(records)


# CPU-backed Exponential Moving Average (EMA) of trainable LoRA weights (0 GiB GPU VRAM overhead)
EMA_DECAY = 0.998
active_model = ddp.module if WORLD > 1 else ddp
ema_shadow = (
    {k: p.detach().float().cpu().clone() for k, p in active_model.named_parameters() if p.requires_grad}
    if RANK == 0 else {}
)


@torch.no_grad()
def update_ema():
    if RANK != 0:
        return
    for k, p in active_model.named_parameters():
        if p.requires_grad and k in ema_shadow:
            ema_shadow[k].mul_(EMA_DECAY).add_(p.detach().float().cpu(), alpha=1.0 - EMA_DECAY)


def checkpoint_save(step, force_remote=False):
    """All ranks pause while rank0 writes a complete, atomic state snapshot using EMA-stabilized LoRA weights."""
    barrier()
    if RANK == 0:
        folder = CHECKPOINT_DIR / f'checkpoint-{step:09d}'
        temp = CHECKPOINT_DIR / f'.checkpoint-{step:09d}.tmp'
        if temp.exists():
            shutil.rmtree(temp)
        temp.mkdir(parents=True)
        (temp/'adapter').mkdir()
        active = ddp.module if WORLD > 1 else ddp
        # Swap EMA-smoothed weights in before saving adapter_model.safetensors, then restore live weights
        live_backup = {}
        with torch.no_grad():
            for k, p in active.named_parameters():
                if p.requires_grad and k in ema_shadow:
                    live_backup[k] = p.detach().cpu().clone()
                    p.data.copy_(ema_shadow[k].to(device=p.device, dtype=p.dtype))
        active.save_pretrained(str(temp/'adapter'), safe_serialization=True)
        with torch.no_grad():
            for k, p in active.named_parameters():
                if k in live_backup:
                    p.data.copy_(live_backup[k].to(device=p.device, dtype=p.dtype))
        del live_backup
        processor.save_pretrained(str(temp/'adapter'))
        torch.save(optimizer.state_dict(), temp/'optimizer.pt')
        torch.save(scheduler.state_dict(), temp/'scheduler.pt')
        torch.save(scaler.state_dict(), temp/'scaler.pt')
        meta = {
            'run_fingerprint': CONFIG['run_fingerprint'],
            'dataset_revision': CONFIG['dataset_revision'],
            'world_size': WORLD,
            'optimizer_step': step,
            'per_rank_row_offset': step * LOCAL_PER_STEP,
            'effective_batch_size': WORLD * LOCAL_PER_STEP,
            'all_unique_rows': PLAN['unique_training_records'],
            'saved_at_utc': __import__('datetime').datetime.now(__import__('datetime').timezone.utc).isoformat(),
        }
        (temp/'state.json').write_text(json.dumps(meta, indent=2), 'utf-8')
        if folder.exists():
            shutil.rmtree(folder)
        os.replace(temp, folder)
        temp_latest = CHECKPOINT_DIR/'latest.json.tmp'
        temp_latest.write_text(json.dumps({'path':str(folder),'step':step},indent=2), 'utf-8')
        os.replace(temp_latest, CHECKPOINT_DIR/'latest.json')
        print(f'Saved complete checkpoint: {folder}', flush=True)
        remote_due = force_remote or (step // int(CONFIG['save_steps'])) % int(CONFIG['remote_sync_every_n_saves']) == 0
        if CONFIG['push_checkpoints'] and remote_due:
            from huggingface_hub import HfApi
            write_token = os.environ.get('OPENKHMER_HF_WRITE_TOKEN')
            if not write_token:
                raise RuntimeError('Hugging Face checkpoint backup is enabled but write token missing')
            api = HfApi(token=write_token)
            api.create_repo(CONFIG['checkpoint_repo'], repo_type='model', private=True, exist_ok=True)
            if not api.repo_info(CONFIG['checkpoint_repo'],repo_type='model').private:
                raise RuntimeError('Refusing to push checkpoint to public repository')
            api.upload_folder(
                repo_id=CONFIG['checkpoint_repo'], repo_type='model',
                folder_path=str(folder),
                path_in_repo=f'runs/{CONFIG["run_id"]}/checkpoint-latest',
                commit_message=f'OpenKhmer v7 full corpus step {step}',
            )
            print('Synced latest checkpoint to private HF repository.', flush=True)
        # Limit Kaggle output usage: keep only two fully-written local checkpoints.
        versions = sorted(CHECKPOINT_DIR.glob('checkpoint-*'))
        for old in versions[:-2]:
            if old != folder and old.is_dir():
                shutil.rmtree(old)
    barrier()


BUDGET_SECONDS = float(CONFIG['session_hours']) * 3600
STEP_LIMIT = int(CONFIG['session_steps'])
LOG_EVERY = int(CONFIG['logging_steps'])
SAVE_EVERY = int(CONFIG['save_steps'])
TIME_PAD = float(CONFIG['time_reserve_seconds'])
t0 = time.monotonic()
optimizer.zero_grad(set_to_none=True)

for step in range(step_start, TOTAL_STEPS):
    batch_loss = 0.0
    for micro in range(ACCUM):
        batch = load_batch()
        sync = ddp.no_sync() if WORLD > 1 and micro < ACCUM - 1 else contextlib.nullcontext()
        with sync:
            with torch.autocast('cuda', dtype=torch.float16):
                result = ddp(**batch)
                if result.loss is None:
                    raise RuntimeError('Model returned no supervised OCR loss')
                loss = result.loss / ACCUM
            if not torch.isfinite(loss.detach()).item():
                raise RuntimeError(f'Nonfinite OCR loss at optimizer step {step + 1}')
            scaler.scale(loss).backward()
        batch_loss += float(loss.detach().item())
        del batch, result, loss
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(trainable, float(CONFIG['max_grad_norm']))
    old_scale = scaler.get_scale()
    scaler.step(optimizer)
    scaler.update()
    scheduler.step()
    optimizer.zero_grad(set_to_none=True)
    update_ema()
    completed = step + 1
    if RANK == 0 and (completed % LOG_EVERY == 0 or completed == step_start + 1):
        sec = time.monotonic() - t0
        new = completed - step_start
        images = new * WORLD * LOCAL_PER_STEP
        speed = images / max(sec, 1.0)
        remaining = (TOTAL_STEPS - completed) * WORLD * LOCAL_PER_STEP
        print(f'STEP {completed:,}/{TOTAL_STEPS:,}  loss={batch_loss:.4f}  '
              f'{speed:.2f} images/s  time={sec/3600:.2f} h  '
              f'epoch ETA={remaining/max(speed,1e-8)/3600:.1f} h  '
              f'GPU0 memory={torch.cuda.max_memory_allocated(DEVICE)/2**30:.2f} GiB', flush=True)
        if scaler.get_scale() < old_scale:
            print('  FP16 overflow detected; optimizer update may have been skipped.', flush=True)

    reached_time = (time.monotonic() - t0) >= max(60.0, BUDGET_SECONDS - TIME_PAD)
    reached_test_steps = STEP_LIMIT > 0 and (completed - step_start) >= STEP_LIMIT
    should_stop = sync_stop(reached_time or reached_test_steps or completed == TOTAL_STEPS)
    should_save = completed % SAVE_EVERY == 0 or should_stop
    if should_save:
        checkpoint_save(completed, force_remote=should_stop)
    if should_stop:
        info('Finished this Kaggle session at step', completed,
             '| Complete epoch?', completed == TOTAL_STEPS)
        break

info('Session complete. Rank0 saved the latest adapter and optimizer to', CHECKPOINT_DIR)
barrier()
if WORLD > 1:
    dist.destroy_process_group()
'''
HELPER_PATHS = {
    'openkhmer_stream_core.py':OPENKHMER_STREAM_CORE,
    'openkhmer_ddp_plan.py':OPENKHMER_DDP_PLAN,
    'openkhmer_ddp_runner.py':OPENKHMER_DDP_RUNNER,
}
for name,source in HELPER_PATHS.items():
    target=WORK_DIR/name
    target.write_text(source,encoding='utf-8')
    py_compile.compile(str(target),doraise=True)
    print('Helper ready:',target,'bytes:',target.stat().st_size)

## 5 — Index **EVERY** eligible training label at a fixed Hugging Face revision

This reads the remote `train_list.txt` records and metadata, not 100GB of images. Default checks the completed 630-TAR export inventory. As your upload was still in progress in earlier checks, **wait until it completes** before claiming a full run. If it is incomplete, this cell stops instead of silently training a subset.

**PIN THE PRINTED SHA** in the Section 2 configuration cell when restarting from a later Kaggle session. Otherwise future uploads could change the stream order and make a checkpoint unsafe to resume.


In [ ]:
from openkhmer_stream_core import prepare_index, EXPECTED_EXPORT_TARS

EXPECTED_TRAIN_SOURCE_ROWS = {
  "math_recognition/krotreaksmey_synthetic_khmer_math-523870ac07": 117515,
  "recognition/EmHengly-Kaggle-d7ccb07b22": 124782,
  "recognition/KimkosalYon_khmer-ocr-200k-siemreap-arial-3362dc3c9f": 182334,
  "recognition/KiteAether_c_khmer_gemma_ocr_train-40487e8a1f": 6363,
  "recognition/KrorngAI_WildKhmerST-Cropped-fc0467951c": 22986,
  "recognition/LazyGreed_khmer_img2txt_200K-1ea579cc51": 147588,
  "recognition/Sokheng_khmer-synthetic-ocr-v1-100k-8277ebd2bb": 60724,
  "recognition/SoyVitou_62k-images-khmer-printed-dataset-bad7f101af": 41351,
  "recognition/SoyVitou_KhmerSynthetic1M-680e080d2f": 568681,
  "recognition/SoyVitou_khmer-handwritten-dataset-4.2k-d5753479c9": 2951,
  "recognition/SoyVitou_khmer_font_printed_dataset-43b26fdf55": 42184,
  "recognition/mrrtmob_khmer_english_ocr_image_line-38c2033c24": 9151339,
  "recognition/seanghay_khmer-hanuman-100k-d571cc00d4": 94060,
  "recognition/vichetkao_khmer_word_dataset-fe255908b1": 22851
}
assert sum(EXPECTED_TRAIN_SOURCE_ROWS.values()) == 10_585_709
if USE_EXISTING_FULL_LOCAL_MIRROR:
    os.environ['OPENKHMER_LOCAL_REPO_DIR'] = str(FULL_LOCAL_MIRROR)
else:
    os.environ.pop('OPENKHMER_LOCAL_REPO_DIR', None)

corpus = prepare_index(
    repo_id=HF_DATASET_REPO,
    token=HF_TOKEN,
    revision=HF_DATASET_REVISION,
    index_dir=INDEX_DIR,
    include_math=INCLUDE_MATH_RECOGNITION,
    allowlist=APPROVED_SOURCES,
    require_complete=REQUIRE_COMPLETE_UPLOAD,
    expected_export_tars=EXPECTED_EXPORT_TARS,
    seed=3407,
)

expected = {k:v for k,v in EXPECTED_TRAIN_SOURCE_ROWS.items()
            if INCLUDE_MATH_RECOGNITION or k.startswith('recognition/')}
if APPROVED_SOURCES is not None:
    expected = {k:v for k,v in expected.items() if k in APPROVED_SOURCES}
for source,expected_rows in expected.items():
    observed = corpus['source_rows'].get(source)
    if observed != expected_rows:
        raise RuntimeError(f'Missing/mismatched train examples for {source}: {observed} != {expected_rows}')
if set(corpus['source_rows']) != set(expected):
    raise RuntimeError(f'Unexpected/missing sources: {set(corpus["source_rows"]) ^ set(expected)}')

print('FIXED DATASET REVISION:', corpus['revision'])
print('After kernel restart, paste that SHA into HF_DATASET_REVISION in the Section 2 configuration cell.')
print(f'Exact eligible train samples indexed: {corpus["total_rows"]:,}')
print(f'TAR archives selected for model training: {corpus["train_shards"]:,}')
print('All 630 uploaded TARs:', corpus['total_repo_tars'])
print('Source row counts:')
for name,n in sorted(corpus['source_rows'].items()):
    print(f' {name}: {n:,}')
assert corpus['total_rows'] == sum(expected.values())


## 6 — Partition the complete corpus across **T4 GPU 0** and **T4 GPU 1**

All selected training TARs are assigned once, to exactly one of the two workers. The two streams are balanced by image count; the shorter stream reuses only a small tail-padding portion if necessary so DDP workers perform the same number of optimizer updates. A repeat count is printed—**there is no hidden sample truncation**.


In [ ]:
from openkhmer_ddp_plan import partition_full_corpus
plan = partition_full_corpus(
    corpus, PLAN_DIR,
    world_size=WORLD_SIZE,
    batch_per_gpu=PER_GPU_BATCH,
    accumulation=GRAD_ACCUM_STEPS,
    epochs=EPOCHS,
)
print('DDP rank rows:', [f'{x:,}' for x in plan['per_rank_unique_records']])
print('Training images including small DDP padding:',
      f'{plan["per_rank_training_presentations"]*WORLD_SIZE:,}')
print('Rank padding:',plan['padded_presentations_per_rank'])
print('Each rank sees distinct source TARs, exactly once per epoch.')
print('TOTAL optimizer steps:', f'{plan["total_optimizer_steps"]:,}')
print('Global effective batch:',plan['effective_batch_size'])
assert plan['unique_training_records']==corpus['total_rows']*EPOCHS


## 7 — Set up fingerprinted checkpoints and the 100-step benchmark

Do not change your pinned source revision, LoRA settings, batch/accumulation or image resize after the first saved checkpoint. With `SESSION_STEPS=100`, the trainer stops and saves at step 100; then change just `SESSION_STEPS=0` in the Section 2 configuration cell and rerun that cell, then Sections 7–8 to continue. If a private checkpoint repo is configured, the final checkpoint at a session boundary is uploaded there.


In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download

FINGERPRINT_OPTIONS = {
    'plan_fingerprint': plan['plan_fingerprint'],
    'model_id': MODEL_ID, 'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY, 'warmup_steps': WARMUP_STEPS,
    'lora_r': LORA_R, 'lora_alpha': LORA_ALPHA,
    'finetune_vision_layers': False,  # Pinned to False so RUN_ID stays d84cd55f108195b1 when upgrading at step 107,744
    'gradient_checkpointing': GRADIENT_CHECKPOINTING,
    'max_image_side': MAX_IMAGE_SIDE, 'max_seq_length': MAX_SEQ_LENGTH,
    'seed':3407,
}
RUN_FINGERPRINT=hashlib.sha256(json.dumps(FINGERPRINT_OPTIONS,sort_keys=True).encode()).hexdigest()
RUN_ID=RUN_FINGERPRINT[:16]
CHECKPOINT_DIR=WORK_DIR/f'openkhmer_v7_checkpoints_{RUN_ID}'
CHECKPOINT_DIR.mkdir(parents=True,exist_ok=True)

resume_path = Path(RESUME_CHECKPOINT_PATH) if RESUME_CHECKPOINT_PATH else None
if RESTORE_CHECKPOINT_FROM_HF:
    if resume_path is not None:
        raise RuntimeError('Choose RESTORE_CHECKPOINT_FROM_HF or RESUME_CHECKPOINT_PATH, not both')
    recovery_dir=WORK_DIR/f'openkhmer_v7_restored_{RUN_ID}'
    recovery_dir=Path(snapshot_download(
        repo_id=CHECKPOINT_REPO,repo_type='model',token=HF_TOKEN,
        allow_patterns=[f'runs/{RUN_ID}/checkpoint-latest/**'],
        local_dir=str(recovery_dir),
    ))
    resume_path=recovery_dir/'runs'/RUN_ID/'checkpoint-latest'

if resume_path is None and (CHECKPOINT_DIR/'latest.json').is_file():
    resume_path=Path(json.loads((CHECKPOINT_DIR/'latest.json').read_text('utf-8'))['path'])
if resume_path:
    state=json.loads((resume_path/'state.json').read_text('utf-8'))
    if state.get('run_fingerprint') != RUN_FINGERPRINT:
        raise RuntimeError('Refusing checkpoint from a different model/dataset/batch/image config')
    print('Resume from verified checkpoint:',resume_path,'step',state['optimizer_step'])
else:
    print('No prior checkpoint; first run starts at step 0.')

run_config = {
    'model_id':MODEL_ID,
    'dataset_revision':corpus['revision'],
    'plan_path':str(PLAN_DIR/'ddp_plan.json'),
    'corpus_index_dir':str(INDEX_DIR),
    'world_size':WORLD_SIZE,
    'batch_per_gpu':PER_GPU_BATCH,
    'gradient_accumulation':GRAD_ACCUM_STEPS,
    'learning_rate':LEARNING_RATE,
    'weight_decay':WEIGHT_DECAY,
    'warmup_steps':WARMUP_STEPS,
    'lora_r':LORA_R,'lora_alpha':LORA_ALPHA,
    'finetune_vision_layers':FINETUNE_VISION_LAYERS,
    'gradient_checkpointing':GRADIENT_CHECKPOINTING,
    'max_image_side':MAX_IMAGE_SIDE,
    'max_seq_length':MAX_SEQ_LENGTH,
    'max_grad_norm':MAX_GRAD_NORM,
    'seed':3407,
    'prefetch_queue':PREFETCH_IMAGES_PER_GPU,
    'local_repo_dir':str(FULL_LOCAL_MIRROR) if USE_EXISTING_FULL_LOCAL_MIRROR else None,
    'cache_tars_on_disk':CACHE_ACTIVE_TAR_TO_TMP,
    'max_single_tar_gb':MAX_SINGLE_TAR_GB,
    'scratch_dir':str(SCRATCH_DIR),
    'session_steps':SESSION_STEPS,
    'session_hours':MAX_SESSION_HOURS,
    'time_reserve_seconds':STOP_RESERVE_SECONDS,
    'save_steps':SAVE_EVERY_STEPS,
    'logging_steps':LOG_EVERY_STEPS,
    'checkpoint_dir':str(CHECKPOINT_DIR),
    'resume_path':str(resume_path) if resume_path else None,
    'run_fingerprint':RUN_FINGERPRINT,
    'run_id':RUN_ID,
    'push_checkpoints':PUSH_CHECKPOINTS_TO_HF,
    'remote_sync_every_n_saves':REMOTE_SYNC_EVERY_N_SAVES,
    'checkpoint_repo':CHECKPOINT_REPO,
}
RUN_CONFIG_PATH=WORK_DIR/f'openkhmer_v7_run_{RUN_ID}.json'
RUN_CONFIG_PATH.write_text(json.dumps(run_config,indent=2),encoding='utf-8')
print('Run ID:',RUN_ID)
print('Run fingerprint:',RUN_FINGERPRINT)
print('Benchmark optimizer steps this session:',SESSION_STEPS if SESSION_STEPS else 'time budget')
print('Checkpoint directory:',CHECKPOINT_DIR)
print('Private HF automatic backup:',PUSH_CHECKPOINTS_TO_HF)

## 8 — START both T4 GPUs (real `torchrun` workers)

**Only run after verifying source rights.** This cell launches exactly two GPU workers when `REQUIRE_DUAL_T4=True`. The GPU workers print which T4 they each own. During training look for `images/s` and `STEP` logs, and for both `GPU 0` and `GPU 1` utilization in Kaggle's GPU monitor.

If DDP/Unsloth vision raises a compatibility error, don't retry a large run repeatedly: copy the first traceback so we can target the exact library issue.


In [ ]:
if not ACKNOWLEDGE_SOURCE_RIGHTS:
    raise RuntimeError(
        'The 2026-10-21 export lists source training approval as pending. '
        'Review exact rights/provenance for the intended research or launch use, '
        'restrict APPROVED_SOURCES accordingly, then explicitly acknowledge it.'
    )

RUNNER_FILE=WORK_DIR/'openkhmer_ddp_runner.py'
assert RUNNER_FILE.is_file() and RUN_CONFIG_PATH.is_file()
env=os.environ.copy()
env['OPENKHMER_V7_RUN_CONFIG']=str(RUN_CONFIG_PATH)
env['OPENKHMER_HF_TOKEN']=HF_TOKEN
env['TOKENIZERS_PARALLELISM']='false'
env.setdefault('OMP_NUM_THREADS','2')
env.setdefault('NCCL_P2P_DISABLE','1')  # More robust on T4 pairings with no P2P link.
if HF_WRITE_TOKEN:
    env['OPENKHMER_HF_WRITE_TOKEN']=HF_WRITE_TOKEN
if USE_EXISTING_FULL_LOCAL_MIRROR:
    env['OPENKHMER_LOCAL_REPO_DIR']=str(FULL_LOCAL_MIRROR)
else:
    env.pop('OPENKHMER_LOCAL_REPO_DIR',None)

if WORLD_SIZE==2:
    cmd=[sys.executable,'-m','torch.distributed.run','--standalone',
         '--nproc_per_node=2',str(RUNNER_FILE)]
else:
    cmd=[sys.executable,str(RUNNER_FILE)]
print('Starting real DDP:', ' '.join(cmd),flush=True)
with subprocess.Popen(cmd, cwd=str(WORK_DIR), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
    for line in proc.stdout:
        print(line, end='', flush=True)
    if proc.wait() != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)

## 9 — Verify checkpoints and continue after the first 100 steps

The **100-step benchmark does not stop full-corpus training permanently**. After checking the speed and loss, edit the Section 2 configuration cell to `SESSION_STEPS=0`, **rerun that cell**, then rerun **Sections 7 and 8** (and earlier setup only if needed). It detects the local checkpoint automatically. On a **new Kaggle session**, paste the pinned dataset SHA from Cell 5, attach the prior Kaggle checkpoint as a dataset, or enable `RESTORE_CHECKPOINT_FROM_HF=True` if you used private backup. Rebuild exactly the same source index and rank partition before resuming.

Full `/tmp` dataset mirrors are temporary; keep checkpoints outside `/tmp`. For an incomplete 26-hour quota, your *checkpoint* is your deliverable, not a finished 10.59-million-image epoch.


In [ ]:
from pathlib import Path
import json
print('Work directory:',WORK_DIR)
print('Checkpoint root:',CHECKPOINT_DIR)
latest=CHECKPOINT_DIR/'latest.json'
if latest.exists():
    state=json.loads(latest.read_text('utf-8'))
    path=Path(state['path'])
    metadata=json.loads((path/'state.json').read_text('utf-8'))
    print('Latest saved optimizer step:',metadata['optimizer_step'])
    print('Per-GPU consumed image presentations:',metadata['per_rank_row_offset'])
    print('Total global images presented:',metadata['per_rank_row_offset'] * WORLD_SIZE)
    print('Adapter:',path/'adapter')
    print('Optimizer state:',path/'optimizer.pt')
    print('Pinned HF dataset SHA:',metadata['dataset_revision'])
else:
    print('No checkpoint saved yet. Run the training cell first.')


## Practical notes for the remaining Kaggle GPU quota

1. **First run:** `SESSION_STEPS=100` → measure actual images/second on **T4×2** and verify the checkpoint saves. A run speed measured with 768px images and your current source mix is more trustworthy than extrapolating from the old 500-step notebook.
2. **Continue:** `SESSION_STEPS=0` → resume from the saved checkpoint in this session. The full corpus remains indexed and eligible, not capped to the 100-step benchmark.
3. **If CUDA OOM:** change `PER_GPU_BATCH=1` and `GRAD_ACCUM_STEPS=4` **before a fresh run**. A checkpoint made with a different batch configuration is intentionally rejected.
4. **If image IO stalls GPU:** try `CACHE_ACTIVE_TAR_TO_TMP=True` (one local TAR per rank). If your original complete `/tmp/openkhmer_hf_full` mirror fits and exists, use `USE_EXISTING_FULL_LOCAL_MIRROR=True` instead. Do not download all 100GB blindly.
5. **If Khmer CER is poor:** 768px and a frozen vision encoder are speed-oriented; try 896px or unfreeze vision in a **new experiment**, not by modifying the existing checkpoint. Keep validation/test out of training.
6. **Quotas:** Kaggle GPU time and session duration are limited; 26 hours total cannot guarantee one complete pass of 10.59M images. Configure remote/private checkpoint persistence if you plan to train beyond a session, and verify the repo is private before using it.

**Official references:** [Unsloth multi-GPU DDP](https://unsloth.ai/docs/basics/multi-gpu-training-with-unsloth), [Unsloth vision fine-tuning](https://unsloth.ai/docs/basics/vision-fine-tuning), [PyTorch DDP no_sync](https://docs.pytorch.org/docs/stable/generated/torch.nn.parallel.DistributedDataParallel.html).
